# Anchor genes — TFs co-expressed with SEN102 (preliminary)

Single-bait guilt-by-association ranking, per **D_GENE_COEX.md §6**.

**Bait:** SEN102 = EGI `8058001` (PCD-execution C1A protease).
**Candidates:** the ~1.7k EGI transcription factors (TFDB).
**Resource:** ATTED-II `…subagging.z.d` — precomputed, condition-independent
co-expression neighborhoods (subagging z-transformed Mutual-Rank index).

> **Blindness (§2).** This notebook is hypothesis-agnostic: it opens *only* SEN102's
> neighborhood + the full TF universe. It never queries the D locus (`Sobic.006g147400`
> / EGI `8084661`), the chr6 interval, or the NAC family. Unblinding is a separate,
> later step — do not add it here.

This preliminary pass reports the **forward** ranking (TFs ranked by their co-expression
score within SEN102's genome-wide neighborhood). Reciprocal-rank / empirical-FDR
refinement (§6.1 steps 2–4) is the next iteration.


In [1]:
from pathlib import Path
import pandas as pd

# --- Frozen configuration (D_GENE_COEX.md §6.1) ---------------------------------
ROOT    = Path("/Users/daffa/workspace/infobio")
ZD      = ROOT / "Sbi-r.c1-0" / "Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d"
TF_EGI  = ROOT / "thesis" / "resources" / "TFDB" / "Sbi_TF_gene_ids_EGI.txt"

SEN102  = "8058001"          # bait — the ONLY gene id hard-coded here
# NOTE: D (8084661) / chr6 / NAC are deliberately absent. Keep it that way.

assert (ZD / SEN102).exists(), "SEN102 neighborhood file missing"
print("resource :", ZD.name)
print("bait     : SEN102 =", SEN102)


resource : Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d
bait     : SEN102 = 8058001


In [2]:
# --- Load the TF universe (EGI ids) ---------------------------------------------
tf_egi = pd.read_csv(TF_EGI)["GeneID"].astype(str).str.strip()
tf_set = set(tf_egi)
print(f"TF universe: {len(tf_set)} EGI ids")

# --- Load SEN102's genome-wide co-expression neighborhood -----------------------
# file format: <partner_EGI>\t<coex_score>, sorted descending by score
sen = pd.read_csv(
    ZD / SEN102, sep="\t", header=None, names=["EGI", "coex_score"], dtype={"EGI": str}
)
sen["rank_genomewide"] = range(1, len(sen) + 1)   # 1-based rank within SEN102's list
N_total = len(sen)
print(f"SEN102 neighborhood: {N_total} partner genes")


TF universe: 1706 EGI ids
SEN102 neighborhood: 21626 partner genes


In [3]:
# --- Restrict SEN102's neighborhood to the TF universe, rank --------------------
tf_coex = (
    sen[sen["EGI"].isin(tf_set)]
    .copy()
    .reset_index(drop=True)
)
# percentile = how high this TF sits within SEN102's full genome-wide neighborhood
tf_coex["percentile"] = 1 - tf_coex["rank_genomewide"] / N_total
tf_coex["rank_among_TFs"] = range(1, len(tf_coex) + 1)
tf_coex = tf_coex[["rank_among_TFs", "EGI", "coex_score", "rank_genomewide", "percentile"]]

n_present = len(tf_coex)
print(f"{n_present} / {len(tf_set)} TFs appear in SEN102's neighborhood "
      f"({n_present/len(tf_set):.0%})")
print(f"coex_score range among TFs: {tf_coex.coex_score.min():.3f} … "
      f"{tf_coex.coex_score.max():.3f}")


1154 / 1706 TFs appear in SEN102's neighborhood (68%)
coex_score range among TFs: -2.695 … 3.776


In [19]:
# --- Top TFs co-expressed with SEN102 (the preliminary deliverable) -------------
TOP = 10
tf_coex.head(TOP)
for row in tf_coex["EGI"].head(TOP):
    print(row)


8084681
8057408
110433436
8068166
8079064
8084661
8078009
8069679
8077856
110431541


In [ ]:
# # --- Persist the full ranked table (D-blind; no unblinding here) -----------------
# OUT = Path("/Users/daffa/workspace/infobio/thesis/analysis/prototype/tf_coex_with_SEN102.tsv")
# tf_coex.to_csv(OUT, sep="\t", index=False)
# print(f"wrote {len(tf_coex)} ranked TFs -> {OUT}")


## Expanding the anchor dPCD genes set

| Generic Gene name | Arabidopsis gene ID, as found in the dPCD paper | Homolog in sorghum |
|-------------------|-------------------------------------------------|--------------------|
| PASPA3 - Aspartic protease 3 | At4g04460 | LOC8058198	LOC110430277	LOC110433911 |
| CEP1 - Cysteine endopeptidase | At5g50260 | LOC8058001 |
| MC9 - Metacaspase | At5g04200 | N/A |
| SCPL48 - Serine carboxypeptidase-like | At3g45010 | LOC8074232 |
| BFN1 - Bifunctional nuclease | At1g11190 | N/A |
| RNS3 - T2 Ribonuclease | At1g26820 | LOC8063515	LOC8063317	LOC8077825 | 

- Based on the Orvela-Carillo (2015) paper. Interestingly, the paper also mentioned that NAC family TFs are also included in this genes set. Utilizing this information, in the TF gene filtering, we will only take into account NAC family TF from the 2000 genes.
- Homolog(s) in sorghum is detected by ATTED-II built in orthologs tab



### dPCD genes set
8058198
110430277
110433911
8058001
8074232
8063515
8063317
8077825


### TF genes (top 10)
8084681
8057408
110433436
8068166
8079064
8084661
8078009
8069679
8077856
110431541